# 06 · Simulación de producción

El modelo no recibe tráfico real, así que se simula con los días **posteriores a la fecha de corte del entrenamiento** (13 de octubre):

- **18 lotes diarios**, del 14 al 31 de octubre. Un lote = la cartera en mora de un día, que es la lista que recibe el call center cada mañana (patrón batch).
- **Horizonte:** el target de un lote (¿paga en 7 días?) llega **7 días después**. Por eso, al procesar el lote del día *d* se evalúa el desempeño del lote *d − 7*. Los lotes del 25 al 31 no tienen target dentro de octubre.
- **Deriva inducida** desde el lote del **18 de octubre**: se simula un error de integración con el sistema de cartera. `dias_mora` se desordena entre los créditos del lote (se rompe su relación con el pago) y se le suman 90 días (cambia su distribución). Los lotes del 14 al 17 no se alteran.

Este notebook arma las tablas `REFERENCIA` y `LOTES` en Snowflake. El flujo de Kestra `produccion_lotes.yml` las procesa lote por lote.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from monitoreo.deriva import TRAMOS_MORA, psi
from src.datos import borrar_tabla, escribir_tabla
from src.lotes import INICIO_DERIVA, construir_tablas

## 1. Reiniciar la simulación

Se borran las tablas de resultados de una corrida anterior, para que la simulación empiece desde cero.

In [3]:
for tabla in ["PREDICCIONES", "MONITOREO", "ALERTAS"]:
    borrar_tabla(tabla)
print("Tablas de resultados borradas")

Tablas de resultados borradas


## 2. Referencia y lotes

Cuántos créditos tiene cada lote, cuántos tienen target y cuáles están alterados.

In [4]:
referencia, lotes = construir_tablas()
print(f"Referencia (entrenamiento, 1 al 13 de octubre): {len(referencia):,} filas")
print(f"Lotes (14 al 31 de octubre): {len(lotes):,} filas")

resumen = lotes.groupby("fecha_lote").agg(creditos=("numero_operacion", "size"),
                                          con_target=("paga_7d", "count"),
                                          alterado=("alterado", "max"))
resumen

Referencia (entrenamiento, 1 al 13 de octubre): 36,674 filas
Lotes (14 al 31 de octubre): 57,506 filas


,creditos,con_target,alterado
fecha_lote,,,
2026-10-14,2574,2574,0
2026-10-15,2507,2507,0
2026-10-16,2419,2419,0
2026-10-17,3970,3970,0
2026-10-18,3970,3970,1
2026-10-19,3849,3849,1
2026-10-20,3462,3462,1
2026-10-21,3285,3285,1
2026-10-22,3180,3180,1


## 3. Qué se alteró exactamente

Se compara `dias_mora` en cada lote contra el entrenamiento: promedio y PSI. Antes del 18 de octubre la distribución se parece a la del entrenamiento; desde el 18 se desplaza unos 90 días.

In [ ]:
comparacion = pd.DataFrame({
    fecha: {"dias_mora_promedio": lote["dias_mora"].mean(),
            "psi_dias_mora": psi(referencia["dias_mora"], lote["dias_mora"], cortes=TRAMOS_MORA)}
    for fecha, lote in lotes.groupby("fecha_lote")
}).T.round(3)
print(f"Promedio de dias_mora en el entrenamiento: {referencia['dias_mora'].mean():.1f}")
comparacion

Promedio de dias_mora en el entrenamiento: 294.6


,dias_mora_promedio,psi_dias_mora
2026-10-14,327.662,1.318
2026-10-15,337.026,1.434
2026-10-16,349.282,1.286
2026-10-17,213.249,1.222
2026-10-18,304.249,4.396
2026-10-19,311.674,4.364
2026-10-20,336.061,4.264
2026-10-21,349.998,4.212
2026-10-22,358.458,4.149
2026-10-23,368.072,4.121


## 4. Guardar en Snowflake

`REFERENCIA` y `LOTES` se guardan en el esquema `CARTERA.PRODUCCION`.

In [6]:
escribir_tabla(referencia, "REFERENCIA", reemplazar=True)
escribir_tabla(lotes, "LOTES", reemplazar=True)
print("REFERENCIA y LOTES guardadas en CARTERA.PRODUCCION")

REFERENCIA y LOTES guardadas en CARTERA.PRODUCCION


## 5. Falsas alarmas: por qué dias_mora usa tramos de mora

Se calcula el PSI de **cada variable** en los lotes sin alterar (14 al 17). Para `dias_mora` se compara además el PSI con los 10 grupos de deciles contra el PSI con los **tramos de mora que usa el negocio** (1-30, 31-60, 61-90, 91-120, 121-180, 181-360 y más de 360 días).

In [7]:
from monitoreo.deriva import comparar
from src.pipeline import CATEGORICAS, NUMERICAS

categoricas = CATEGORICAS + ["plazo_terminado", "compromiso_semana_previa"]
normales = lotes[lotes["alterado"] == 0]
psi_normales = pd.DataFrame({
    fecha: comparar(referencia, lote, NUMERICAS, categoricas).set_index("variable")["psi"]
    for fecha, lote in normales.groupby("fecha_lote")
}).round(3)

tramos = [0, 30, 60, 90, 120, 180, 360, float("inf")]
def psi_tramos(ref, act):
    p_ref = pd.cut(ref, tramos).value_counts(normalize=True).clip(lower=1e-4)
    p_act = pd.cut(act, tramos).value_counts(normalize=True).clip(lower=1e-4)
    return float(((p_act - p_ref) * np.log(p_act / p_ref)).sum())

import numpy as np
psi_normales.loc["dias_mora (tramos)"] = [psi_tramos(referencia["dias_mora"], l["dias_mora"])
                                          for _, l in normales.groupby("fecha_lote")]
psi_normales.loc["dias_mora (tramos), lote alterado 18-oct"] = psi_tramos(
    referencia["dias_mora"], lotes.loc[lotes["fecha_lote"] == "2026-10-18", "dias_mora"])
psi_normales.round(3)

,2026-10-14,2026-10-15,2026-10-16,2026-10-17
variable,,,,
dias_mora,1.318,1.434,1.286,1.222
cuotas_mora,0.008,0.014,0.024,0.061
cuotas_por_pagar,0.000,0.001,0.001,0.006
cuotas_pagadas,0.002,0.002,0.003,0.009
total_cuotas,0.000,0.001,0.001,0.003
capital_insoluto,0.001,0.001,0.002,0.011
capital_vencido,0.006,0.011,0.020,0.052
saldo_cuota_mas_vencida,0.005,0.006,0.007,0.021
valor_deuda_exigible,0.009,0.013,0.020,0.058
